In [4]:
import pandas as pd
import numpy as np
import wfdb
import ast
import sklearn
import heartpy as hp

import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MultiLabelBinarizer

In [5]:
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

In [7]:
path = 'ptb-xl/'
sampling_rate=500

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)




KeyboardInterrupt



In [4]:
# Filter stuff
## ECG data typically has low pass filters set to 150 Hz. It helps smooth out the ECG signal but may impact the final amplitudes
Low_cutoff = 150.0

## High pass filters are often 0.5 Hz to cancel "respiration arifacts"
High_cutoff = 0.5

## we could also have power line filters to reduce noise caused by external electronics or anti-aliasing filters to reduce errors when the nyquist frequency is violated, but we are going to assume the data collected was done so expertly. If patients and rooms are prepared properly no filters may be needed but in some cases it is required.
# og_X = np.copy(X)
# for x_ind in range(1):
#     for index in range(1):
#         print(np.transpose(X[x_ind, :,index]))
#         X[x_ind, :, index] = hp.filter_signal(data = np.transpose(X[x_ind, :,index]), cutoff=[High_cutoff, Low_cutoff], sample_rate=sampling_rate, filtertype='bandpass')
#         print(np.transpose(X[x_ind, :, index]))


filt_X = np.copy(X)
for x_ind in range(len(filt_X)):
    for index in range(len(filt_X[x_ind, 0, :])):
        filt_X[x_ind, :, index] = hp.filter_signal(data = np.transpose(X[x_ind, :,index]), cutoff=[High_cutoff, Low_cutoff], sample_rate=sampling_rate, filtertype='bandpass')

In [5]:
np.array_equal(X, filt_X)

False

In [6]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
X_filt_train = filt_X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass

# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
X_filt_test = filt_X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

### Fixing Y data for proper training format

In [7]:
mlb = MultiLabelBinarizer()
y_train_enc = mlb.fit_transform(y_train)
y_test_enc = mlb.fit_transform(y_test)
print("Encoded Labels (y_encoded):\n", y_train_enc)
print("\nClass Names (Order of Columns):\n", mlb.classes_)

Encoded Labels (y_encoded):
 [[0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 ...
 [0 0 0 0 1]
 [0 0 0 1 0]
 [0 0 0 1 0]]

Class Names (Order of Columns):
 ['CD' 'HYP' 'MI' 'NORM' 'STTC']


# Model Training

## Model structure

In [14]:
# THis is the number of classes. For Superclass it will be 5, if we do all of or a subset of the subclasses there can be more or less.
NUM_CLASSES = 5

# The input shape for the dataset
IN_SHAPE = (sampling_rate*10,12)

model = Sequential([
    Input(IN_SHAPE),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='sigmoid')]
)

## Compiling and Fitting (No filtering)

In [15]:
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', # could also use categorical_crossentropy here for a single choice per input. Change output to softmax if doing that approach though
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)]
)

In [16]:
model.fit(X_train, y_train_enc, epochs=20, verbose=1, validation_data=(X_test, y_test_enc))

Epoch 1/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 188s 302ms/step - accuracy: 0.5306 - auc_2: 0.7762 - loss: 0.4451 - val_accuracy: 0.6110 - val_auc_2: 0.8424 - val_loss: 0.3877
Epoch 2/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 179s 292ms/step - accuracy: 0.5992 - auc_2: 0.8489 - loss: 0.3786 - val_accuracy: 0.6015 - val_auc_2: 0.8484 - val_loss: 0.3855
Epoch 3/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 177s 289ms/step - accuracy: 0.6219 - auc_2: 0.8759 - loss: 0.3481 - val_accuracy: 0.5919 - val_auc_2: 0.8477 - val_loss: 0.3773
Epoch 4/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 175s 286ms/step - accuracy: 0.6418 - auc_2: 0.8952 - loss: 0.3228 - val_accuracy: 0.6183 - val_auc_2: 0.8599 - val_loss: 0.3772
Epoch 5/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 176s 287ms/step - accuracy: 0.6640 - auc_2: 0.9135 - loss: 0.2964 - val_accuracy: 0.6233 - val_auc_2: 0.8600 - val_loss: 0.3754
Epoch 6/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 174s 284ms/step - accuracy: 0.6851 - auc_2: 0.9289 - loss: 0.2697 - val_accuracy: 0.6224 - val_auc_2: 0.8541 - val_loss:

## Compiling and Fitting (filtering)

In [17]:
model2 = Sequential([
    Input(IN_SHAPE),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='sigmoid')]
)
model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    # could also use categorical_crossentropy here for a single choice per input. Change output to softmax if doing that approach though
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)]
)

In [18]:
model2.fit(X_filt_train, y_train_enc, epochs=20, verbose=1, validation_data=(X_test, y_test_enc))

Epoch 1/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 184s 295ms/step - accuracy: 0.5380 - auc_3: 0.7942 - loss: 0.4263 - val_accuracy: 0.5496 - val_auc_3: 0.8003 - val_loss: 0.4422
Epoch 2/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 175s 285ms/step - accuracy: 0.6042 - auc_3: 0.8548 - loss: 0.3693 - val_accuracy: 0.5641 - val_auc_3: 0.8262 - val_loss: 0.4265
Epoch 3/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 172s 281ms/step - accuracy: 0.6345 - auc_3: 0.8781 - loss: 0.3409 - val_accuracy: 0.5723 - val_auc_3: 0.8301 - val_loss: 0.4165
Epoch 4/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 171s 279ms/step - accuracy: 0.6600 - auc_3: 0.8971 - loss: 0.3148 - val_accuracy: 0.5551 - val_auc_3: 0.8301 - val_loss: 0.4261
Epoch 5/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 174s 284ms/step - accuracy: 0.6813 - auc_3: 0.9141 - loss: 0.2897 - val_accuracy: 0.5823 - val_auc_3: 0.8290 - val_loss: 0.4164
Epoch 6/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 179s 291ms/step - accuracy: 0.7016 - auc_3: 0.9293 - loss: 0.2640 - val_accuracy: 0.5805 - val_auc_3: 0.8329 - val_loss:

## Testing

In [19]:
loss, accuracy, auc_score = model.evaluate(X_test, y_test_enc, verbose=1)

# Print the results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test AUC: {auc_score:.4f}")

loss2, accuracy2, auc_score2 = model2.evaluate(X_filt_test, y_test_enc, verbose=1)

# Print the results
print(f"Test Loss: {loss2:.4f}")
print(f"Test Accuracy: {accuracy2:.4f}")
print(f"Test AUC: {auc_score2:.4f}")

69/69 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.5869 - auc_2: 0.8139 - loss: 0.7186
Test Loss: 0.7186
Test Accuracy: 0.5869
Test AUC: 0.8139
69/69 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - accuracy: 0.6119 - auc_3: 0.8242 - loss: 0.7825
Test Loss: 0.7825
Test Accuracy: 0.6119
Test AUC: 0.8242


In [26]:
from tensorflow.keras.layers import BatchNormalization, Dropout, Input, Add
from tensorflow.keras.models import Model

# Improved model with Batch Normalization and Dropout
NUM_CLASSES = 5
IN_SHAPE = (5000, 12)

def create_improved_cnn():
    inputs = Input(shape=IN_SHAPE)
    
    # First Conv Block
    x = Conv1D(filters=64, kernel_size=7, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Second Conv Block
    x = Conv1D(filters=128, kernel_size=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Third Conv Block
    x = Conv1D(filters=256, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.4)(x)
    
    # Fourth Conv Block
    x = Conv1D(filters=256, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.4)(x)
    
    # Global Average Pooling (better than Flatten)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output layer
    outputs = Dense(NUM_CLASSES, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create the improved model
model_v2 = create_improved_cnn()

In [27]:
from tensorflow.keras.optimizers import Adam
# Use a lower learning rate for better convergence
optimizer = Adam(learning_rate=0.0005)

model_v2.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True, name='auc')]
)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Early stopping - stop if validation loss doesn't improve
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when validation loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

# Save best model
checkpoint = ModelCheckpoint(
    'best_ecg_model.keras',
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stop, reduce_lr, checkpoint]

print("Starting training with improved model...")
print("="*60)

history = model_v2.fit(
    X_filt_train, 
    y_train_enc,
    validation_data=(X_filt_test, y_test_enc),
    epochs=50,  # Will stop early if no improvement
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


Starting training with improved model...
Epoch 1/50
583/613 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.4420 - auc: 0.6912 - loss: 0.5898

In [ ]:
loss, accuracy, auc_score = model_v2.evaluate(X_test, y_test_enc, verbose=1)

print(f"\n RESULTS:")
print(f"   Test Loss: {loss:.4f}")
print(f"   Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Test AUC: {auc_score:.4f}")
